In [1]:
import sqlite3
import pandas as pd
from datetime import datetime, timedelta

# Kết nối database trong bộ nhớ (In-memory)
conn = sqlite3.connect(':memory:')
cursor = conn.cursor()

# Tạo bảng student
cursor.execute('''
CREATE TABLE student (
    student_id INTEGER PRIMARY KEY,
    name TEXT,
    class TEXT,
    course_id INTEGER,
    score REAL
)
''')

# Tạo bảng course
cursor.execute('''
CREATE TABLE course (
    id INTEGER PRIMARY KEY,
    course_name TEXT
)
''')

# Chèn dữ liệu vào bảng student (dựa trên ảnh)
students_data = [
    (1, 'Nguyen Minh Hoang', 'May Tinh', 12, 6.7),
    (2, 'Tran Thi Lan', 'Kinh Te', 34, 9.2),
    (3, 'Pham Van Nam', 'Toan Tin', None, 7.9),
    (4, 'Le Thanh Huyen', 'Toan Tin', 20, 7.2),
    (5, 'Vu Quoc Anh', 'May Tinh', 24, 8.0),
    (6, 'Dang Thuy Linh', 'May Tinh', 24, 5.5),
    (7, 'Bui Tien Dung', 'Kinh Te', 34, 9.2),
    (8, 'Ho Ngoc Mai', 'Toan Tin', 20, 8.8),
    (9, 'Duong Huu Phuc', 'Kinh Te', None, 7.2),
    (10, 'Cao Thi Hanh', 'May Tinh', None, 7.0)
]
cursor.executemany('INSERT INTO student VALUES (?,?,?,?,?)', students_data)

# Chèn dữ liệu vào bảng course
courses_data = [
    (12, 'Giai tich'),
    (34, 'Thong ke'),
    (26, 'Tin hoc')
]
cursor.executemany('INSERT INTO course VALUES (?,?)', courses_data)

def show_query(query, title):
    print(f"\n--- {title} ---")
    display(pd.read_sql_query(query, conn))

print("Dữ liệu ban đầu đã sẵn sàng!")

Dữ liệu ban đầu đã sẵn sàng!


In [2]:
# Tích Descartes (Cross Join)
show_query("SELECT * FROM student CROSS JOIN course", "1.1 Tích Descartes")

# INNER JOIN
show_query("SELECT * FROM student INNER JOIN course ON student.course_id = course.id", "1.2 INNER JOIN")

# LEFT JOIN
show_query("SELECT * FROM student LEFT JOIN course ON student.course_id = course.id", "1.3 LEFT JOIN")

# Lưu ý: SQLite không hỗ trợ trực tiếp RIGHT JOIN và FULL OUTER JOIN
# nhưng ta có thể mô phỏng bằng LEFT JOIN


--- 1.1 Tích Descartes ---


,student_id,name,class,course_id,score,id,course_name
0,1,Nguyen Minh Hoang,May Tinh,12.0,6.7,12,Giai tich
1,1,Nguyen Minh Hoang,May Tinh,12.0,6.7,26,Tin hoc
2,1,Nguyen Minh Hoang,May Tinh,12.0,6.7,34,Thong ke
3,2,Tran Thi Lan,Kinh Te,34.0,9.2,12,Giai tich
4,2,Tran Thi Lan,Kinh Te,34.0,9.2,26,Tin hoc
5,2,Tran Thi Lan,Kinh Te,34.0,9.2,34,Thong ke
6,3,Pham Van Nam,Toan Tin,NaN,7.9,12,Giai tich
7,3,Pham Van Nam,Toan Tin,NaN,7.9,26,Tin hoc
8,3,Pham Van Nam,Toan Tin,NaN,7.9,34,Thong ke
9,4,Le Thanh Huyen,Toan Tin,20.0,7.2,12,Giai tich



--- 1.2 INNER JOIN ---


,student_id,name,class,course_id,score,id,course_name
0,1,Nguyen Minh Hoang,May Tinh,12,6.7,12,Giai tich
1,2,Tran Thi Lan,Kinh Te,34,9.2,34,Thong ke
2,7,Bui Tien Dung,Kinh Te,34,9.2,34,Thong ke



--- 1.3 LEFT JOIN ---


,student_id,name,class,course_id,score,id,course_name
0,1,Nguyen Minh Hoang,May Tinh,12.0,6.7,12.0,Giai tich
1,2,Tran Thi Lan,Kinh Te,34.0,9.2,34.0,Thong ke
2,3,Pham Van Nam,Toan Tin,NaN,7.9,NaN,None
3,4,Le Thanh Huyen,Toan Tin,20.0,7.2,NaN,None
4,5,Vu Quoc Anh,May Tinh,24.0,8.0,NaN,None
5,6,Dang Thuy Linh,May Tinh,24.0,5.5,NaN,None
6,7,Bui Tien Dung,Kinh Te,34.0,9.2,34.0,Thong ke
7,8,Ho Ngoc Mai,Toan Tin,20.0,8.8,NaN,None
8,9,Duong Huu Phuc,Kinh Te,NaN,7.2,NaN,None
9,10,Cao Thi Hanh,May Tinh,NaN,7.0,NaN,None


In [3]:
# Cập nhật course_id còn thiếu (ví dụ điền môn 26)
cursor.execute("UPDATE student SET course_id = 26 WHERE course_id IS NULL")

# Loại bỏ bản ghi có môn học không tồn tại trong bảng course
cursor.execute("DELETE FROM student WHERE course_id NOT IN (SELECT id FROM course)")

# a. Thống kê theo Lớp
show_query("""
    SELECT class, COUNT(*) as total_students, AVG(score) as avg_score
    FROM student GROUP BY class
""", "2a. Thống kê theo Lớp")

# b. Thống kê theo Môn học
show_query("""
    SELECT c.course_name, COUNT(s.student_id) as total_students, AVG(s.score) as avg_score
    FROM student s JOIN course c ON s.course_id = c.id
    GROUP BY c.course_name
""", "2b. Thống kê theo Môn học")

# c. Phân loại theo điểm trung bình môn học
# Bước này dùng CASE WHEN để phân loại
show_query("""
    SELECT c.course_name, AVG(s.score) as avg_score,
    CASE
        WHEN AVG(s.score) >= 9.0 THEN 'Xuat sac'
        WHEN AVG(s.score) >= 6.0 THEN 'Tot'
        ELSE 'Kem'
    END as classification
    FROM student s JOIN course c ON s.course_id = c.id
    GROUP BY c.course_name
""", "2c. Phân loại môn học")


--- 2a. Thống kê theo Lớp ---


,class,total_students,avg_score
0,Kinh Te,3,8.533333
1,May Tinh,2,6.850000
2,Toan Tin,1,7.900000



--- 2b. Thống kê theo Môn học ---


,course_name,total_students,avg_score
0,Giai tich,1,6.700000
1,Thong ke,2,9.200000
2,Tin hoc,3,7.366667



--- 2c. Phân loại môn học ---


,course_name,avg_score,classification
0,Giai tich,6.700000,Tot
1,Thong ke,9.200000,Xuat sac
2,Tin hoc,7.366667,Tot


In [5]:
# a, b, c. Xếp hạng
ranking_query = """
SELECT name, class, course_id, score,
       RANK() OVER (ORDER BY score DESC) as rank_all,
       RANK() OVER (PARTITION BY class ORDER BY score DESC) as rank_class,
       RANK() OVER (PARTITION BY course_id ORDER BY score DESC) as rank_course
FROM student
"""
show_query(ranking_query, "3. Xếp hạng sinh viên")


--- 3. Xếp hạng sinh viên ---


,name,class,course_id,score,rank_all,rank_class,rank_course
0,Tran Thi Lan,Kinh Te,34,9.2,1,1,1
1,Bui Tien Dung,Kinh Te,34,9.2,1,1,1
2,Pham Van Nam,Toan Tin,26,7.9,3,1,1
3,Duong Huu Phuc,Kinh Te,26,7.2,4,3,2
4,Cao Thi Hanh,May Tinh,26,7.0,5,1,3
5,Nguyen Minh Hoang,May Tinh,12,6.7,6,2,1


In [4]:
# Thêm cột mới
try:
    cursor.execute("ALTER TABLE student ADD COLUMN graduation_date DATETIME")
except:
    pass # Tránh lỗi nếu chạy lại cell nhiều lần

# Cập nhật ngày tốt nghiệp dựa trên hạng điểm số
cursor.execute("""
    WITH RankedStudent AS (
        SELECT student_id, RANK() OVER (ORDER BY score DESC) as rnk FROM student
    )
    UPDATE student
    SET graduation_date = datetime('now', '+' || (SELECT rnk FROM RankedStudent WHERE RankedStudent.student_id = student.student_id) || ' days')
""")

show_query("SELECT name, score, graduation_date FROM student ORDER BY score DESC", "4. Ngày tốt nghiệp")


--- 4. Ngày tốt nghiệp ---


,name,score,graduation_date
0,Tran Thi Lan,9.2,2026-04-04 12:23:42
1,Bui Tien Dung,9.2,2026-04-04 12:23:42
2,Pham Van Nam,7.9,2026-04-06 12:23:42
3,Duong Huu Phuc,7.2,2026-04-07 12:23:42
4,Cao Thi Hanh,7.0,2026-04-08 12:23:42
5,Nguyen Minh Hoang,6.7,2026-04-09 12:23:42


In [8]:
!git clone https://github.com/nguyenanhthumain/K17_SQL.git

Cloning into 'K17_SQL'...
remote: Enumerating objects: 6, done.
remote: Counting objects: 100% (1/1), done.
remote: Total 6 (delta 0), reused 0 (delta 0), pack-reused 5 (from 1)
Receiving objects: 100% (6/6), 85.74 KiB | 28.58 MiB/s, done.
